# SquirrelBGone — YOLOv8s Multi-Class Training (v2)

Trains a YOLOv8**s** model on a dataset containing both squirrel and bird classes,
so bird detections can be suppressed instead of misclassified as squirrels.

**Required classes:** squirrel, bird (others like cat, dog, raccoon are a bonus)

**Why YOLOv8s over n:** better multi-class precision at similar Pi 5 inference cost.
Swap `yolov8s.pt` → `yolov8n.pt` in Step 5 if inference is too slow after deployment.

**Before running:** Runtime → Change runtime type → T4 GPU

## Step 1 — Check GPU

In [ ]:
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none — go to Runtime > Change runtime type > T4 GPU'}")

## Step 2 — Install dependencies

In [ ]:
%pip install -q ultralytics roboflow
print("Done.")

## Step 3 — Find a suitable dataset version

Probes Warren Wiens 'squirrel-detector-1.1' versions 1–7 for one that includes
both `squirrel` and `bird`. If none is found, follow **Step 3b** to supply an
alternative dataset.

Get a free Roboflow API key at https://app.roboflow.com/settings/api

In [ ]:
from roboflow import Roboflow
from pathlib import Path
import yaml

ROBOFLOW_API_KEY = "rf_your_key_here"  # paste your key here

REQUIRED = {"squirrel", "bird"}

rf      = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("warren-wiens-d0d4p").project("squirrel-detector-1.1")
dataset = None

for v in range(1, 8):
    try:
        print(f"Trying version {v}...", end=" ", flush=True)
        ds = project.version(v).download(
            "yolov8", location=f"/content/dataset_v{v}", overwrite=True
        )
        with open(Path(ds.location) / "data.yaml") as f:
            classes = set(yaml.safe_load(f)["names"])
        print(f"classes: {sorted(classes)}")
        if REQUIRED.issubset(classes):
            print(f"  ✓ version {v} has required classes — using this one")
            dataset = ds
            break
        else:
            print(f"  ✗ missing: {REQUIRED - classes}")
    except Exception as e:
        print(f"not available")
        break

if dataset is None:
    print("""
No version of 'squirrel-detector-1.1' has both squirrel and bird classes.
Proceed to Step 3b to supply an alternative dataset, then continue from Step 4.
""")
else:
    print(f"\nDataset location: {dataset.location}")

## Step 3b — Alternative dataset (run only if Step 3 found nothing)

Find a Roboflow Universe dataset that has **both** `squirrel` and `bird` annotations,
paste its workspace/project/version below, and run this cell.

**How to find one:**
1. Go to roboflow.com/universe and search for `squirrel bird`
2. Open a dataset, confirm the class list includes both, note the URL slug
   (e.g. `roboflow.com/<workspace>/<project>/<version>/...`)
3. Fill in the three variables below

**Alternative — build one yourself in Roboflow (free):**
1. Create a new project, upload the Warren Wiens squirrel images
2. Fork any public bird detection dataset into the same project
3. Export as YOLOv8, fill in the slug below

In [ ]:
# Only run this cell if Step 3 did not find a suitable dataset
ALT_WORKSPACE = ""   # e.g. "my-workspace"
ALT_PROJECT   = ""   # e.g. "squirrels-and-birds"
ALT_VERSION   = 1

if not ALT_WORKSPACE or not ALT_PROJECT:
    print("Fill in ALT_WORKSPACE and ALT_PROJECT above, then re-run this cell.")
else:
    project_alt = rf.workspace(ALT_WORKSPACE).project(ALT_PROJECT)
    dataset     = project_alt.version(ALT_VERSION).download("yolov8")

    with open(Path(dataset.location) / "data.yaml") as f:
        _d = yaml.safe_load(f)
    print("Classes:", _d["names"])
    missing = REQUIRED - set(_d["names"])
    if missing:
        raise ValueError(f"Still missing required classes: {missing}")
    print(f"✓ Required classes present. Dataset: {dataset.location}")

## Step 4 — Inspect dataset

In [ ]:
data_yaml = Path(dataset.location) / "data.yaml"
with open(data_yaml) as f:
    data = yaml.safe_load(f)

print("Classes:", data["names"])
print("Num classes:", data["nc"])

for split in ["train", "valid", "test"]:
    img_dir = Path(dataset.location) / split / "images"
    if img_dir.exists():
        print(f"{split}: {len(list(img_dir.glob('*.*')))} images")

## Step 5 — Train YOLOv8s

- 75 epochs with patience=15 early stopping
- `batch=16` is safe for T4; try `batch=32` if no OOM
- Expect ~35–45 minutes on T4

To fall back to the faster nano model, change `yolov8s.pt` → `yolov8n.pt` below.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")  # small — better multi-class accuracy than nano

results = model.train(
    data=str(data_yaml),
    epochs=75,
    imgsz=640,
    batch=16,
    name="squirrelbgone_v2",
    patience=15,
    save=True,
    plots=True,
    device=0,
    workers=2,
)

print("\nTraining complete.")
print(f"Best weights: {results.save_dir}/weights/best.pt")

## Step 6 — Evaluate on test set

In [ ]:
best_model = YOLO(f"{results.save_dir}/weights/best.pt")

metrics = best_model.val(
    data=str(data_yaml),
    split="test",
)

print(f"mAP50:     {metrics.box.map50:.3f}")
print(f"mAP50-95:  {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")

print("\nPer-class AP50:")
per_class = dict(zip(data["names"], metrics.box.ap50))
for name, ap in sorted(per_class.items(), key=lambda x: -x[1]):
    print(f"  {name:<12} {ap:.3f}")

print("\n--- Key classes ---")
for cls in ["squirrel", "bird"]:
    ap   = per_class.get(cls, 0)
    flag = "✓" if ap >= 0.70 else "⚠ low — consider more training data for this class"
    print(f"  {cls:<12} AP50={ap:.3f}  {flag}")

## Step 7 — Download best.pt

In [ ]:
import shutil
from google.colab import files

best_pt = f"{results.save_dir}/weights/best.pt"
dest    = "/content/squirrelbgone_v2_best.pt"
shutil.copy(best_pt, dest)

print(f"Model size: {Path(best_pt).stat().st_size / 1e6:.1f} MB")
print("Downloading...")
files.download(dest)

## Step 8 — Smoke test (optional)

Run a few test images and confirm squirrel and bird both appear in predictions.

In [ ]:
import glob
from IPython.display import Image, display

test_images = glob.glob(f"{dataset.location}/test/images/*.*")[:4]
if not test_images:
    print("No test images found.")
else:
    for img_path in test_images:
        result = best_model.predict(img_path, conf=0.35, save=True,
                                    project="/content", name="smoke_test",
                                    exist_ok=True)
        annotated = (glob.glob("/content/smoke_test/*.jpg") +
                     glob.glob("/content/smoke_test/*.png"))
        if annotated:
            display(Image(sorted(annotated)[-1], width=640))
        for r in result:
            for box in r.boxes:
                print(f"  {data['names'][int(box.cls)]}: {float(box.conf):.2f}")
        print()

## Deployment

1. Move the downloaded file into the repo: `models/squirrelbgone_v2_best.pt`
2. Commit and push
3. On the Pi: `git pull`
4. In `.env`, set: `MODEL_PATH=models/squirrelbgone_v2_best.pt`
5. Restart `detect.py`

The multi-class model will log `class=bird` for bird detections.
`BENIGN_CLASSES` in `detect.py` suppresses them from triggering the sprayer.

**If inference is too slow on Pi 5**, retrain with `yolov8n.pt` in Step 5.